<h1 style="text-align: center;">Physical AI의 Vision-LLM 융합 시청각 멀티모달 시스템</h1>

<br><br>

<div style="text-align: right; color: gray; font-style: italic;">
강사 김규래<br>
kkr.kyurae.kim@gmail.com
</div><br>

---
---


## 6. Vision-LLM 융합 멀티모달 시스템

### A. Vision-to-Text

#### $1)$ YOLO 탐지 결과 구조

In [ ]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO


model = YOLO("src/models/YOLO/yolo11n_int8.engine")

results = model.predict(
    source="src/images/city.png",
    conf=0.25,
    iou=0.5,
    classes=None,
)

yolo_test_img = results[0].plot()

yolo_test_img_rgb = cv2.cvtColor(yolo_test_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12,8))
plt.imshow(yolo_test_img_rgb)
plt.axis("off");

In [ ]:
print(results[0].boxes.xyxy)

print(results[0].boxes.conf)

print(results[0].boxes.cls)

#### $2)$ YOLO 결과를 JSON 구조로 변환

원하는 구조:
```python
{
    "image_width": 1280,
    "image_height": 720,
    "objects": [
        {
            "class": "person",
            "confidence": 0.92,
            "bbox": {
                "x1": 100,
                "y1": 80,
                "x2": 450,
                "y2": 700
            }
        }
    ]
}
```

In [ ]:
from ultralytics import YOLO
import cv2
import json


model = YOLO("src/models/YOLO/yolo11n_int8.engine")


image = cv2.imread("src/images/city.png")

height, width = image.shape[:2]

results = model.predict(
    source=image,
    conf=0.25,
    iou=0.5,
    verbose=False,
)

result = results[0]


objects = []

for box in result.boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])
    x1, y1, x2, y2 = (box.xyxy[0].cpu().tolist())

    objects.append(
        {
            "class": result.names[class_id],
            "confidence": round(confidence, 3),
            "bbox": {
                "x1": int(x1),
                "y1": int(y1),
                "x2": int(x2),
                "y2": int(y2),
            }
        }
    )


vision_data = {
    "image_width": width,
    "image_height": height,
    "objects": objects,
}


vision_json = json.dumps(
    vision_data,
    ensure_ascii=False,
    indent=4,
)


print(vision_json)

#### $3)$ JSON - 자연어 변환

단순 Template:
```python
text = (
    f"{class_name} 객체가 "
    f"confidence {confidence:.2f}로 "
    f"탐지되었습니다."
)
```

In [ ]:
def detections_to_text(vision_data):
    objects = vision_data["objects"]

    if len(objects) == 0:
        return "현재 탐지된 객체가 없습니다."

    sentences = []

    for index, obj in enumerate(objects, start=1):
        sentence = f"{index}번 객체는 {obj['class']}이며, confidence는 {obj['confidence']:.2f}입니다."

        sentences.append(sentence)

    return "\n".join(sentences)

In [ ]:
vision_text = detections_to_text(vision_data)

print(vision_text)

#### $3)$ JSON - LLM 연결

JSON 구조를 통해 자연어로 변환된 Vision 결과를 이전 섹션에서 배운 Context로써 Prompt에 추가하여 LLM에 전달합니다.

```text
예시:

Instruction:
주어진 객체 탐지 정보를 바탕으로
현재 상황을 설명하세요.

Context:
1번 객체는 car이며, confidence는 0.93입니다.
2번 객체는 car이며, confidence는 0.88입니다.
3번 객체는 person이며, confidence는 0.81입니다.
4번 객체는 car이며, confidence는 0.41입니다.
5번 객체는 car이며, confidence는 0.40입니다.
6번 객체는 potted plant이며, confidence는 0.38입니다.

Constraint:
탐지 결과에 없는 객체를 추측하지 마세요.

Output Format:
한국어 두 문장 이내
```

In [ ]:
# TODO: vision_text를 사용하여 LLM에 context 전달


from llama_cpp import Llama


MODEL_PATH = "src/models/Gemma4/google_gemma-4-E2B-it-Q4_K_M.gguf"
CONTEXT_WINDOW = 2048
MAX_TOKENS = 150

llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=CONTEXT_WINDOW,
    n_batch=32,
    n_ubatch=32,
    verbose=False,
)

response = llm.create_chat_completion(
    messages=[
        {
            "role": "system",
            "content": """
                        Instruction:
                        주어진 객체 탐지 정보를 바탕으로 현재 상황을 설명하세요.

                        Constraint:
                        탐지 결과에 없는 객체를 추측하지 마세요.

                        Output Format:
                        한국어 두 문장 이내
                       """
        },
        {
            "role": "user",
            "content": f"""
                        Context:
                        {vision_text}
                       """
        }
    ],
    max_tokens=100,
    temperature=0.7
)


print(response["choices"][0]["message"]["content"])

In [ ]:
# TODO: 카메라를 연결하여 LLM에 Vision Context 전달
# TODO: 실시간 프레임 처리 Loop에서 특정 이벤트(예: if key == ord("l"))를 통해 LLM에 Vision-Context 전달

# 카메라 프레임 전체 화면 YOLO 감지 객체를 JSON으로 Gemma 전달
import cv2

from ultralytics import YOLO
from llama_cpp import Llama


YOLO_MODEL_PATH = "src/models/YOLO/yolo11n.pt"
GEMMA_MODEL_PATH = "src/models/Gemma4/google_gemma-4-E2B-it-Q4_K_M.gguf"
CONTEXT_WINDOW = 2048
MAX_TOKENS = 150


yolo = YOLO(YOLO_MODEL_PATH)
yolo.to("cuda")


llm = Llama(
    model_path=GEMMA_MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=CONTEXT_WINDOW,
    n_batch=32,
    n_ubatch=32,
    verbose=False,
)


pipeline = (
    "nvarguscamerasrc sensor-id=0 ! "
    "video/x-raw(memory:NVMM), "
    "width=1280, height=720, framerate=30/1 ! "
    "nvvidconv ! "
    "video/x-raw, format=BGRx ! "
    "videoconvert ! "
    "video/x-raw, format=BGR ! "
    "queue leaky=downstream max-size-buffers=1 ! "
    "appsink drop=true max-buffers=1 sync=false"
)


cap = cv2.VideoCapture(pipeline, cv2.CAP_GSTREAMER)

if not cap.isOpened():
    print("카메라를 열 수 없습니다.")
    exit()


print("q : 종료")
print("l : 현재 YOLO 탐지 결과를 Gemma에게 전달")


while True:
    ret, frame = cap.read()

    if not ret:
        break
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    results = yolo.predict(
        source=frame,
        conf=0.25,
        iou=0.5,
        verbose=False,
    )

    result = results[0]


    vision_text_list = []

    for index, box in enumerate(result.boxes):
        class_id = int(box.cls[0].item())
        confidence = float(box.conf[0].item())
        class_name = yolo.names[class_id]

        text = f"{index + 1}번 객체는 {class_name}이며, confidence는 {confidence:.2f}입니다."
        vision_text_list.append(text)

    if vision_text_list:
        vision_text = " ".join(vision_text_list)
    else:
        vision_text = "현재 탐지된 객체가 없습니다."

    output_frame = result.plot()

    cv2.imshow("YOLO + Gemma", output_frame)

    if cv2.waitKey(1) & 0xFF == ord("l"):
        print("\n[Vision Context]")
        print(vision_text)

        response = llm.create_chat_completion(
            messages=[
                {
                    "role": "system",
                    "content": """
                                Instruction:
                                주어진 객체 탐지 정보를 바탕으로 현재 상황을 설명해.

                                Constraint:
                                탐지 결과에 없는 객체를 추측하지 마.

                                Output Format:
                                한국어 두 문장 이내
                               """
                },
                {
                    "role": "user",
                    "content": f"""
                                Context:
                                {vision_text}
                               """,
                },
            ],
            max_tokens=MAX_TOKENS,
            temperature=0.7
        )

        answer = response["choices"][0]["message"]["content"]

        print("\n[Gemma]")
        print(answer)


cap.release()
cv2.destroyAllWindows()

### B. Image 기반 Vision-Language 멀티모달 시스템

In [ ]:
# city.png 전체를 이미지로 Gemma에 전달


from llama_cpp import Llama
from llama_cpp.llama_chat_format import Gemma4ChatHandler

import cv2
import base64


MODEL_PATH = "src/models/Gemma4/google_gemma-4-E2B-it-Q4_K_M.gguf"
MMPROJ_PATH = "src/models/Gemma4/mmproj-google_gemma-4-E2B-it-f16.gguf"
IMAGE_PATH = "src/images/city.png"
CONTEXT_WINDOW = 2048
MAX_TOKENS = 150


chat_handler = Gemma4ChatHandler(clip_model_path=MMPROJ_PATH)


llm = Llama(
    model_path=MODEL_PATH,
    chat_handler=chat_handler,
    n_gpu_layers=-1,
    n_ctx=CONTEXT_WINDOW,
    n_batch=32,
    n_ubatch=32,
    verbose=False,
)



image = cv2.imread(IMAGE_PATH)

success, buffer = cv2.imencode(".jpg", image)

if not success:
    raise RuntimeError("이미지 인코딩에 실패했습니다.")

image_base64 = base64.b64encode(buffer).decode("utf-8")

image_data = ("data:image/jpeg;base64," + image_base64)


response = llm.create_chat_completion(
    messages=[
        {
            "role": "system",
            "content": """
                        Instruction:
                        주어진 이미지를 바탕으로 현재 상황을 설명해.

                        Constraint:
                        이미지에서 명확하게 확인되지 않는 내용은 추측하지 마.

                        Output Format:
                        한국어 두 문장 이내
                       """
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "현재 카메라 이미지를 설명하시오."
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_data
                    },
                },
            ],
        }
    ],
    max_tokens=MAX_TOKENS,
    temperature=0.7,
)


print(response["choices"][0]["message"]["content"])

In [ ]:
# city.png 중 신호등 ROI를 이미지로 Gemma에 전달


import cv2
import base64

from ultralytics import YOLO
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Gemma4ChatHandler


YOLO_MODEL_PATH = "src/models/YOLO/yolo11n.pt"

GEMMA_MODEL_PATH = "src/models/Gemma4/google_gemma-4-E2B-it-Q4_K_M.gguf"
MMPROJ_PATH = "src/models/Gemma4/mmproj-google_gemma-4-E2B-it-f16.gguf"
IMAGE_PATH = "src/images/city.png"
CONTEXT_WINDOW = 2048
MAX_TOKENS = 20
DISPLAY_WIDTH = 600
DISPLAY_HEIGHT = 600


yolo = YOLO(YOLO_MODEL_PATH)
yolo.to("cuda")


chat_handler = Gemma4ChatHandler(clip_model_path=MMPROJ_PATH)


llm = Llama(
    model_path=GEMMA_MODEL_PATH,
    chat_handler=chat_handler,
    n_gpu_layers=-1,
    n_ctx=CONTEXT_WINDOW,
    n_batch=32,
    n_ubatch=32,
    verbose=False,
)


image = cv2.imread(IMAGE_PATH)

if image is None:
    raise RuntimeError("이미지를 불러올 수 없습니다.")


results = yolo.predict(
    source=image,
    conf=0.03,
    iou=0.5,
    classes=[9],
    verbose=False,
)

result = results[0]


if len(result.boxes) == 0:
    raise RuntimeError("신호등이 탐지되지 않았습니다.")


selected_box = min(result.boxes, key=lambda box: float(box.xyxy[0][1].item()))

confidence = float(selected_box.conf[0].item())

x1, y1, x2, y2 = (selected_box.xyxy[0].cpu().tolist())

x1 = int(x1)
y1 = int(y1)
x2 = int(x2)
y2 = int(y2)


print(f"Traffic Light Confidence: {confidence:.2f}")
print(f"BBox: ({x1}, {y1}) ({x2}, {y2})")


height, width = image.shape[:2]

x1 = max(0, x1)
y1 = max(0, y1)

x2 = min(width, x2)
y2 = min(height, y2)

traffic_light_image = image[y1:y2, x1:x2].copy()

if traffic_light_image.size == 0:
    raise RuntimeError("BBox 이미지를 만들 수 없습니다.")


display_image = cv2.resize(traffic_light_image, (DISPLAY_WIDTH, DISPLAY_HEIGHT), interpolation=cv2.INTER_NEAREST)

cv2.namedWindow("Traffic Light", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Traffic Light", DISPLAY_WIDTH, DISPLAY_HEIGHT)
cv2.imshow("Traffic Light", display_image)

print("q : 이미지 창 닫기")

while True:
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cv2.destroyWindow("Traffic Light")


success, buffer = cv2.imencode(".jpg", traffic_light_image)

if not success:
    raise RuntimeError("이미지 인코딩에 실패했습니다.")

image_base64 = base64.b64encode(buffer).decode("utf-8")

image_data = "data:image/jpeg;base64," + image_base64

response = llm.create_chat_completion(
    messages=[
        {
            "role": "system",
            "content": """
                        Instruction:
                        주어진 신호등 이미지의 현재 색을 판단해.

                        Constraint:
                        반드시 다음 세 가지 중 하나만 대답해.
                        빨간불
                        노란불
                        파란불

                        다른 설명이나 문장을 추가하지 마.

                        Output Format:
                        한 단어
                       """
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "현재 신호등의 색을 판단하시오."
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_data
                    },
                },
            ],
        }
    ],
    max_tokens=MAX_TOKENS,
    temperature=0.0,
)


answer = response["choices"][0]["message"]["content"].strip()

print("\n[Gemma]")
print(answer)

In [ ]:
# TODO: 카메라 프레임 내 YOLO 감지 객체 하나를 선택하여 이미지로 LLM에 전달

# 카메라 프레임 전체 화면 YOLO 감지 객체를 이미지로 Gemma 전달
import cv2
import base64

from ultralytics import YOLO
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Gemma4ChatHandler


YOLO_MODEL_PATH = "src/models/YOLO/yolo11n.pt"
GEMMA_MODEL_PATH = "src/models/Gemma4/google_gemma-4-E2B-it-Q4_K_M.gguf"
MMPROJ_PATH = "src/models/Gemma4/mmproj-google_gemma-4-E2B-it-f16.gguf"
CONTEXT_WINDOW = 2048
MAX_TOKENS = 150


yolo = YOLO(YOLO_MODEL_PATH)
yolo.to("cuda")


chat_handler = Gemma4ChatHandler(clip_model_path=MMPROJ_PATH)


llm = Llama(
    model_path=GEMMA_MODEL_PATH,
    chat_handler=chat_handler,
    n_gpu_layers=-1,
    n_ctx=CONTEXT_WINDOW,
    n_batch=32,
    n_ubatch=32,
    verbose=False,
)


pipeline = (
    "nvarguscamerasrc sensor-id=0 ! "
    "video/x-raw(memory:NVMM), "
    "width=1280, height=720, framerate=30/1 ! "
    "nvvidconv ! "
    "video/x-raw, format=BGRx ! "
    "videoconvert ! "
    "video/x-raw, format=BGR ! "
    "queue leaky=downstream max-size-buffers=1 ! "
    "appsink drop=true max-buffers=1 sync=false"
)


cap = cv2.VideoCapture(pipeline, cv2.CAP_GSTREAMER)

if not cap.isOpened():
    print("카메라를 열 수 없습니다.")
    exit()


print("q : 종료")
print("l : 가장 왼쪽 객체를 Gemma에게 전달")


while True:
    ret, frame = cap.read()

    if not ret:
        break
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    results = yolo.predict(
        source=frame,
        conf=0.25,
        iou=0.5,
        verbose=False,
    )

    result = results[0]


    detected_objects = []

    for box in result.boxes:
        class_id = int(box.cls[0].item())
        confidence = float(box.conf[0].item())
        class_name = yolo.names[class_id]
        x1, y1, x2, y2 = (box.xyxy[0].cpu().tolist())

        detected_objects.append(
            {
                "class_name": class_name,
                "confidence": confidence,
                "bbox": (
                    int(x1),
                    int(y1),
                    int(x2),
                    int(y2),
                ),
            }
        )


    selected_object = None

    if detected_objects:
        selected_object = min(detected_objects, key=lambda x: x["bbox"][0])


    output_frame = result.plot()

    if selected_object is not None:
        x1, y1, x2, y2 = (selected_object["bbox"])
        cv2.rectangle(output_frame, (x1, y1), (x2, y2), (255, 255, 255), 3)

    cv2.imshow("YOLO + Gemma", output_frame)


    if cv2.waitKey(1) & 0xFF == ord("l"):
        if selected_object is None:
            print("\n탐지된 객체가 없습니다.")
            continue

        class_name = selected_object["class_name"]
        confidence = selected_object["confidence"]
        x1, y1, x2, y2 = selected_object["bbox"]

        print("\n[Selected Object]")
        print(f"Class: {class_name}")
        print(f"Confidence: {confidence:.2f}")
        print(f"BBox: ({x1}, {y1}) ({x2}, {y2})")


        height, width = frame.shape[:2]

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(width, x2)
        y2 = min(height, y2)

        object_image = frame[y1:y2, x1:x2].copy()

        if object_image.size == 0:
            print("BBox 이미지를 만들 수 없습니다.")
            continue


        cv2.imshow("Selected Object", object_image)


        success, buffer = cv2.imencode(".jpg", object_image)

        if not success:
            print("이미지 인코딩에 실패했습니다.")
            continue

        image_base64 = (base64.b64encode(buffer).decode("utf-8"))
        image_data = "data:image/jpeg;base64," + image_base64


        response = llm.create_chat_completion(
            messages=[
                {
                    "role": "system",
                    "content": """
                                Instruction:
                                주어진 이미지를 바탕으로 이미지에 보이는 내용을 설명해.

                                Constraint:
                                이미지에서 명확하게 확인되지 않는 내용은 추측하지 마.

                                Output Format:
                                한국어 두 문장 이내
                               """
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": (
                                "YOLO가 객체로 탐지한 영역이야. 이 이미지에 보이는 내용을 설명해줘."
                            ),
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": image_data
                            },
                        },
                    ],
                }
            ],
            max_tokens=MAX_TOKENS,
            temperature=0.7,
        )


        answer = response["choices"][0]["message"]["content"]

        print("\n[Gemma]")
        print(answer)


cap.release()
cv2.destroyAllWindows()

### C. STT, TTS

#### $1)$ `Whisper.cpp` 패키지 설치:

```bash
git clone https://github.com/ggml-org/whisper.cpp.git
```

```bash
sudo apt install -y cmake
export PATH=/usr/local/cuda-12.6/bin:$PATH
export LD_LIBRARY_PATH=/usr/local/cuda-12.6/lib64:$LD_LIBRARY_PATH
cd whisper.cpp
```

1. CPU 버전으로 빌드:

```bash
cmake -B build-cpu

cmake --build build-cpu --target whisper-cli -j4 --config Release
```

2. CUDA 버전으로 빌드:

```bash
cmake -B build -DGGML_CUDA=1

cmake --build build -j4 --config Release
```

빌드가 끝나면 `whisper-cli` 파일이 생성됩니다.

파일 생성 확인:

CPU:
```bash
ls -lh build-cpu/bin/whisper-cli
```

CUDA:
```bash
ls -lh build/bin/whisper-cli
```

한국어 Whisper 모델 다운로드:

```bash
sh ./models/download-ggml-model.sh base
```

#### 간단한 STT 테스트:

```bash
mkdir ../src/audio
```

5초간 음성 녹음:

```bash
arecord \
    -D plughw:3,0 \
    -f S16_LE \
    -r 16000 \
    -c 1 \
    -d 5 \
    ../src/audio/input.wav
```

STT 테스트:

```bash
./build-cpu/bin/whisper-cli \
    -m models/ggml-base.bin \
    -f ../src/audio/input.wav \
    -l ko \
    -nt
```

#### $2)$ `Piper` 패키지 설치:

별도 가상환경 생성:

```bash
deactivate
python3 -m venv .piper_venv
source .piper_venv/bin/activate
```

`Piper` 설치:

```bash
pip install piper-tts
```

한국어 음성 다운로드:

```bash
cd ..
mkdir -p src/models/Piper
python3 -m piper.download_voices \
    ko_KR-kss-medium \
    --data-dir src/models/Piper
```

TTS 테스트:

```bash
python3 -m piper \
    -m src/models/Piper/ko_KR-kss-medium.onnx \
    -f src/audio/response.wav \
    -- \
    "안녕하세요. 반갑습니다."
```

```bash
aplay src/audio/response.wav
```

#### $3)$ Python 환경에서의 STT, TTS

앞서 명령어로 STT와 TTS가 문제 없이 실행되었다면, 이제 Python 코드로 동일한 작업을 수행해보도록 합시다.

#### STT:

In [ ]:
import subprocess


MIC_DEVICE = "plughw:3,0"
AUDIO_FILE = "src/audio/input.wav"
WHISPER_PATH = "whisper.cpp/build-cpu/bin/whisper-cli"
WHISPER_MODEL = "whisper.cpp/models/ggml-base.bin"
RECORD_SECONDS = 5


print("말씀을 시작해 주세요!")


subprocess.run(
    [
        "pasuspender", "--",
        "arecord",
        "-D", MIC_DEVICE,
        "-f", "S16_LE",
        "-r", "16000",
        "-c", "1",
        "-d", str(RECORD_SECONDS),
        AUDIO_FILE,
    ],
    check=True,
)

result = subprocess.run(
    [
        WHISPER_PATH,
        "-m", WHISPER_MODEL,
        "-f", AUDIO_FILE,
        "-l", "ko",
        "--no-timestamps",
    ],
    text=True,
    capture_output=True,
    check=True,
)


text = result.stdout.strip()

print("STT 결과:")
print(text)

#### TTS:

In [ ]:
import subprocess


PIPER_PYTHON = ".piper_venv/bin/python"
PIPER_MODEL = "src/models/Piper/ko_KR-kss-medium.onnx"
OUTPUT_FILE = "src/audio/response.wav"
SPEAKER_DEVICE = "plughw:2,0"


text = "현재는 빨간불입니다. 주의하시길 바랍니다!"

subprocess.run(
    [
        PIPER_PYTHON,
        "-m",
        "piper",

        "-m",
        PIPER_MODEL,

        "-f",
        OUTPUT_FILE,

        "--",
        text,
    ],
    check=True,
)

subprocess.run(
    [
        "aplay",
        "-D",
        SPEAKER_DEVICE,
        OUTPUT_FILE,
    ],
    check=True,
)

동일한 코드를 함수로 정의하도록 합시다.

In [ ]:
def speech_to_text():
    subprocess.run(
        [
            "pasuspender", "--",
            "arecord",
            "-D", MIC_DEVICE,
            "-f", "S16_LE",
            "-r", "16000",
            "-c", "1",
            "-d", str(RECORD_SECONDS),
            AUDIO_FILE,
        ],
        check=True,
    )

    result = subprocess.run(
        [
            WHISPER_PATH,
            "-m", WHISPER_MODEL,
            "-f", AUDIO_FILE,
            "-l", "ko",
            "--no-timestamps",
        ],
        text=True,
        capture_output=True,
        check=True,
    )


    return result.stdout.strip()

In [ ]:
def text_to_speech(text):
    subprocess.run(
        [
            PIPER_PYTHON,
            "-m",
            "piper",

            "-m",
            PIPER_MODEL,

            "-f",
            OUTPUT_FILE,

            "--",
            text,
        ],
        check=True,
    )

    subprocess.run(
        [
            "aplay",
            "-D",
            SPEAKER_DEVICE,
            OUTPUT_FILE,
        ],
        check=True,
    )

In [ ]:
question = speech_to_text()
print(question)
text_to_speech("앞에 사람이 한 명 있습니다.")

---

## <center>< Section Project ></center>

전 과정에서 다룬 내용을 바탕으로 Vision-LLM 융합 멀티모달 시스템을 구축합니다.<br>
각 섹션의 메인 주제인 Gemma와 YOLO, 그리고 모든 기법들을 통합하여, 객체를 탐지하고 특정 이벤트를 감지하고 유저 음성 입력을 받아 음성으로 내보내는 파이프라인을 설계합니다.

In [ ]:
# TODO: Section 6 "Vision-LLM Multimodal Systems" Project

---
---

<br><br><div style="text-align: right; color: gray; font-style: italic;">
© 2026, 김규래 (Kyu Rae Kim), All rights reserved.&emsp;<br><br>
This material is provided solely for the intended instructional purpose.&emsp;<br>
Redistribution, reproduction, modification, adaptation, or reuse of this material in any form without prior written permission from the copyright holder is prohibited.&emsp;
</div>